# Week 5 — Classification Models (Part 1)

Week 4 measured what ignorance is worth: a `DummyClassifier` that never looks at
a feature scores **4.55%** (1/22) under 5-fold stratified cross-validation. That
is the floor.

This notebook puts the first three real algorithms on top of it:

* **Logistic regression** — a linear model; weighted sums of the features,
  turned into 22 probabilities by softmax.
* **k-nearest neighbours** — a distance-based model; look up the most similar
  training fields and take a vote.
* **Gaussian naive Bayes** — a probabilistic model; Bayes' rule plus one
  deliberately wrong independence assumption.

All three are trained through the same three-line loop, measured on the *same*
cross-validation folds, and collected in one results table. That table is
**extended** in Weeks 6-8, never replaced.

What this notebook does not do: tune a single hyperparameter (Week 6), build an
ensemble (Week 6), explain which features matter (Week 7), or open
`data/processed/test.csv` (Week 8).


## 0. Setup

Same pattern as Weeks 1-4: repository root onto `sys.path`, then import the
logic from `src/` instead of writing it inline. This week's new module is
`src/models/classical_models.py` (`get_logistic_regression`, `get_knn`,
`get_naive_bayes`), covered by `tests/test_classical_models.py`.

Everything else is already built: the Week 3 preprocessor, the Week 4 evaluation
helpers, and `data/processed/train.csv`. The 440 test rows stay untouched.


In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline

from src.data import DEFAULT_RANDOM_STATE, EXPECTED_LABEL_COUNT, FEATURE_COLUMNS, TARGET_COLUMN
from src.evaluation import DEFAULT_CV_FOLDS, cross_validated_accuracy, evaluate_model
from src.models import get_baseline_model, get_knn, get_logistic_regression, get_naive_bayes
from src.preprocessing import build_preprocessor

pd.set_option("display.width", 120)

print("seed", DEFAULT_RANDOM_STATE, "| folds", DEFAULT_CV_FOLDS, "| classes", EXPECTED_LABEL_COUNT)

seed 42 | folds 5 | classes 22


In [2]:
FEATURES = list(FEATURE_COLUMNS)

train = pd.read_csv(REPO_ROOT / "data" / "processed" / "train.csv")
X_train = train[FEATURES]
y_train = train[TARGET_COLUMN]

assert X_train.shape == (1_760, 7)
assert y_train.nunique() == EXPECTED_LABEL_COUNT

print("training rows:", len(train), "| features:", len(FEATURES))

training rows: 1760 | features: 7


## 1. The training loop, named once

Every model from here to Week 12 is trained with the same two calls. Learn the
shape now and nothing later is new:

```python
model.fit(X_train, y_train)      # learn from labelled rows
model.predict(X_test)            # answer for rows it has never seen
```

That is the whole scikit-learn estimator API for supervised learning:
`fit(X, y)` learns and stores parameters on the object, `predict(X)` applies
them. A random forest, a gradient booster and a tuned pipeline are all driven
by exactly these two methods — which is what makes swapping algorithms a
one-word change rather than a rewrite.

Two rules travel with it:

* `fit` sees **training rows only**. Anything it learns from a row the model is
  later scored on is data leakage (Weeks 2-3).
* The factories in `src/models/` return **unfitted** objects, so the caller
  decides what they are fitted on — and cross-validation can fit a fresh clone
  inside every fold.

Here it is, once, in full.


In [3]:
demo_model = get_naive_bayes()
demo_model.fit(X_train, y_train)                 # 1. learn
demo_predictions = demo_model.predict(X_train)   # 2. answer

print("model :", demo_model)
print("first five predictions:", [str(label) for label in demo_predictions[:5]])
print("first five true labels:", list(y_train[:5]))
print("training accuracy     :", round(evaluate_model(demo_model, X_train, y_train)["accuracy"], 4))

model : GaussianNB()
first five predictions: ['orange', 'grapes', 'kidneybeans', 'mothbeans', 'orange']
first five true labels: ['orange', 'grapes', 'kidneybeans', 'mothbeans', 'orange']
training accuracy     : 0.9949


That accuracy is measured on the rows the model just learned from, so it is not
evidence of anything — a model can memorise. Every number quoted from §5 onward
comes from cross-validation, where each score is computed on rows the model in
question did not see.

### The models travel inside the Week 3 pipeline

Two of this week's three algorithms care about the *units* of the features, so
they get the Week 3 `ColumnTransformer` in front of them:

```python
Pipeline([("preprocess", build_preprocessor()), ("model", model)])
```

A `Pipeline` is itself an estimator: `fit` fits the scaler and then the model,
`predict` transforms and then predicts. That matters for cross-validation — the
scaler is re-fitted inside every fold, on that fold's training rows only, so no
validation row's mean or standard deviation ever reaches the model that scores
it. Scaling the whole training set once, before splitting, would leak.


In [4]:
def make_pipeline(model) -> Pipeline:
    """Week 3's preprocessor followed by a Week 5 model, as one estimator."""
    return Pipeline([("preprocess", build_preprocessor()), ("model", model)])


pipeline = make_pipeline(get_logistic_regression())
pipeline.fit(X_train, y_train)
print(pipeline.named_steps)
print("prediction for the first field:", pipeline.predict(X_train.head(1))[0])

{'preprocess': ColumnTransformer(transformers=[('numeric', StandardScaler(),
                                 ['N', 'P', 'K', 'temperature', 'humidity',
                                  'ph', 'rainfall'])],
                  verbose_feature_names_out=False), 'model': LogisticRegression(max_iter=1000, random_state=42)}
prediction for the first field: orange


## 2. Logistic regression — a linear decision boundary

Despite the name, logistic regression is a **classifier**. For each of the 22
crops it computes a weighted sum of the seven features,

```
score(crop) = w1*N + w2*P + w3*K + w4*temperature + w5*humidity + w6*ph + w7*rainfall + b
```

and converts the 22 scores into probabilities with the **softmax** function
(exponentiate, then divide by the total), so they are positive and sum to 1. The
predicted crop is the one with the highest probability. Training chooses the
weights that make the observed training labels as probable as possible.

Because every score is linear in the features, the surface separating any two
crops is **flat**: a line in two dimensions, a plane in three, a hyperplane in
seven. Logistic regression can only draw straight cuts through feature space. If
the true boundary curves, no amount of data will fix that — a different model
family will.

**Softmax vs. one-vs-rest.** Two ways to make a two-class method handle 22
classes: fit a single model over all classes at once (multinomial / softmax), or
fit 22 separate "is it rice or not?" models and take the most confident answer
(one-vs-rest, OvR). scikit-learn 1.6 uses softmax here because the default
`lbfgs` solver supports it; OvR is available via `OneVsRestClassifier` and is
what you fall back to for methods that cannot do better.

`C` is the **inverse** regularisation strength: small `C` pulls the weights
towards zero (a simpler, flatter model), large `C` lets them grow to fit the
training data more closely. It is left at 1.0 this week — choosing it by trying
values and keeping the best is exactly what Week 6 is for.

The cell below also calls a third method: `predict_proba(X)`. Where
`predict` returns one label per row, `predict_proba` returns the 22
probabilities behind that label — one per class, in `classes_` order, each row
summing to 1. Every classifier this week provides it, and `predict` is simply
the class with the largest of those numbers.


In [5]:
logistic = make_pipeline(get_logistic_regression()).fit(X_train, y_train)
coefficients = logistic.named_steps["model"].coef_

print("coefficient matrix:", coefficients.shape, "= (22 crops, 7 features)")
print("intercepts        :", logistic.named_steps["model"].intercept_.shape)
print("learned numbers   :", coefficients.size + logistic.named_steps["model"].intercept_.size)

probabilities = logistic.predict_proba(X_train.head(1))[0]
top = np.argsort(probabilities)[::-1][:3]
print("\nmost probable crops for the first field:")
for index in top:
    print(f"  {logistic.classes_[index]:<12} {probabilities[index]:.3f}")
print("probabilities sum to", round(probabilities.sum(), 6))

coefficient matrix: (22, 7) = (22 crops, 7 features)
intercepts        : (22,)
learned numbers   : 176

most probable crops for the first field:
  orange       0.852
  coconut      0.044
  mungbean     0.044
probabilities sum to 1.0


176 numbers is the entire model: 22 x 7 weights and 22 intercepts. Nothing else
is stored — the training rows themselves are discarded once fitting is done.
Compare that with the next algorithm, which stores all 1,760 of them.

Reading the weights (which crop is pushed up by rainfall, which by potassium) is
Week 7's subject; note only that they *can* be read. That is a real advantage of
linear models and a reason to keep one in the comparison even if it does not
win.


## 3. k-nearest neighbours — prediction by similarity

KNN barely trains at all. `fit` stores the training rows; that is its entire
learning. `predict` does the work: for a new field, measure the distance to
every stored field, keep the `k` closest, and return the most common label among
them. It is the standard example of a **lazy learner** — cost deferred from
training to prediction.

There is no equation of a boundary here. The boundary is implied by the data,
and it can be any shape at all — which is why KNN handles curved class regions
that logistic regression cannot, and why it can never tell you *why* it answered
as it did beyond "the neighbours said so".

### The effect of k

`k` is a smoothness dial:

* `k = 1` — the boundary follows every training point exactly. Training accuracy
  is a perfect and utterly uninformative 100%, and one mislabelled row owns its
  whole neighbourhood. This is the overfitting end.
* moderate `k` — votes average over a small region, so single odd rows are
  outvoted.
* very large `k` — the neighbourhood grows until it covers most of the data and
  the model converges on "predict the most common class", i.e. the Week 4
  baseline. This is the underfitting end.

The sweep below is a *demonstration* of that curve, not a hyperparameter search:
the value is not adopted, and picking `k` properly — with a search, on a stated
protocol — is Week 6.


In [6]:
k_values = [1, 3, 5, 11, 25, 51, 101, 201, 401]
k_sweep = pd.DataFrame(
    [
        {
            "k": k,
            "cv accuracy": round(
                cross_validated_accuracy(make_pipeline(get_knn(n_neighbors=k)), X_train, y_train)[
                    "mean"
                ],
                4,
            ),
        }
        for k in k_values
    ]
).set_index("k")

k_sweep

,cv accuracy
k,
1,0.9665
3,0.9642
5,0.9653
11,0.9534
25,0.9267
51,0.8705
101,0.7710
201,0.6602
401,0.5409


Accuracy is flat for small `k`, then falls away as the neighbourhood swells: at
`k = 401` each vote is drawn from nearly a quarter of the training set, and
crops that occupy small regions of feature space are simply outvoted. Continue
far enough and the curve would arrive at 4.55%.

### Scaling, and why KNN is the model that cares most

KNN's answer is entirely determined by distances, and distances mix all seven
columns into one number. `K` spans about 200 units, `ph` about 6 — so on raw
data a one-unit change in `ph` is invisible next to potassium, whatever it means
agronomically. Standardising first (Week 3) puts every feature on the same
footing, which is the *principled* default and the reason the preprocessor is in
front of every model here.

Principled is not the same as always-better on one dataset:


In [7]:
scaled = cross_validated_accuracy(make_pipeline(get_knn()), X_train, y_train)["mean"]
unscaled = cross_validated_accuracy(get_knn(), X_train, y_train)["mean"]

print("KNN, standardised features:", round(scaled, 4))
print("KNN, raw features         :", round(unscaled, 4))

KNN, standardised features: 0.9653
KNN, raw features         : 0.9767


The raw-feature version scores slightly *higher* here. That is not evidence
against scaling; it is evidence that on this dataset the raw units happen to
weight the most discriminative features (rainfall, humidity, `K`) more heavily,
by accident. The gap is about a percentage point and within a couple of standard
deviations of the fold spread — a difference this small is not yet a difference
(Week 4 §6). Keeping the scaler keeps the comparison between algorithms rather
than between accidental weightings, and it is the choice that generalises to the
next dataset.

### The curse of dimensionality, briefly

Distance-based reasoning degrades as columns are added. In high dimensions the
distances between points concentrate: the nearest neighbour ends up almost as
far away as the farthest, so "nearest" stops implying "similar", and every extra
uninformative column dilutes the informative ones. It is why KNN struggles on
data with hundreds or thousands of features unless something reduces them first.

Seven features is comfortably safe. The effect is easy to *stage*, though —
append 100 columns of pure noise to the seven real ones and watch the same model
collapse:


In [8]:
rng = np.random.default_rng(seed=DEFAULT_RANDOM_STATE)
noise = pd.DataFrame(
    rng.normal(0.0, 1.0, (len(X_train), 100)),
    columns=[f"noise_{i}" for i in range(100)],
)
X_noisy = pd.concat([X_train.reset_index(drop=True), noise], axis=1)
noisy_pipeline = Pipeline(
    [("preprocess", build_preprocessor(list(X_noisy.columns))), ("model", get_knn())]
)

clean_knn = cross_validated_accuracy(make_pipeline(get_knn()), X_train, y_train)["mean"]
noisy_knn = cross_validated_accuracy(noisy_pipeline, X_noisy, y_train)["mean"]

print("7 real features   :", round(clean_knn, 4))
print("7 real + 100 noise:", round(noisy_knn, 4))

7 real features   : 0.9653
7 real + 100 noise: 0.2233


The information did not go anywhere — the seven real columns are still there —
but it has been buried in a distance that is now mostly noise. Logistic
regression and naive Bayes are hurt far less by the same columns, because they
weight (or model) each feature separately rather than summing them all into one
distance.

## 4. Gaussian naive Bayes — probability with a wrong assumption

Bayes' rule turns the question round. What we want is `P(crop | field)`; what is
easy to estimate from training data is `P(field | crop)`:

```
P(crop | field)  ∝  P(field | crop) * P(crop)
```

The `∝` means "proportional to": the missing denominator `P(field)` is the
same for all 22 crops, so it cannot change which of them is largest and is
dropped. `P(crop)` is just the class frequency (1/22 here — the classes are
balanced).
`P(field | crop)` is a seven-dimensional joint distribution, which is hard. So
naive Bayes assumes the features are **independent given the class**, letting
that joint probability factorise into a product of seven one-dimensional ones,
and **Gaussian** naive Bayes models each of those as a normal curve.

Fitting therefore reduces to computing a mean and a variance per feature per
class: 22 x 7 x 2 = 308 numbers, in a single pass over the data. There is no
iterative optimisation and no distance computation, which is why it is the
fastest of the three by a wide margin.


In [9]:
naive_bayes = make_pipeline(get_naive_bayes()).fit(X_train, y_train)
nb_model = naive_bayes.named_steps["model"]

print("class means    :", nb_model.theta_.shape, "= (22 crops, 7 features)")
print("class variances:", nb_model.var_.shape)
print("class priors   :", np.round(nb_model.class_prior_[:3], 4), "... all equal, as expected")

# The assumption is false on this data: Week 2 measured strong correlations.
correlations = X_train.corr().abs()
np.fill_diagonal(correlations.values, 0.0)
strongest = correlations.stack().idxmax()
print(f"\nstrongest feature correlation: {strongest[0]}/{strongest[1]} =",
      round(correlations.loc[strongest], 3))

class means    : (22, 7) = (22 crops, 7 features)
class variances: (22, 7)
class priors   : [0.0455 0.0455 0.0455] ... all equal, as expected

strongest feature correlation: P/K = 0.736


`P` and `K` correlate at 0.74, so "independent given the class" is simply not
true here — and the model works anyway. Two reasons this is less paradoxical
than it sounds:

* **Classification needs a ranking, not a calibration.** Only the *largest* of
  the 22 class scores decides the answer. Correlated features effectively get
  counted twice, which makes the winning probability wildly overconfident (often
  0.999-something), but usually leaves the winner unchanged.
* **Few parameters, little variance.** 308 numbers estimated from 1,760 rows are
  all estimated well. A model with a more faithful assumption and far more
  parameters can easily do worse on data this size.

So: trust naive Bayes' predictions more than its probabilities. And treat it as
the "second baseline" — a fast, assumption-driven model that a well-chosen
algorithm ought to beat. On this dataset, as §5 shows, that turns out to be a
demanding bar.

Unlike the other two, this model is *unaffected* by standardisation — rescaling
a column moves every class's mean and variance for that column identically. The
preprocessor stays in front of it only so that all three models receive exactly
the same inputs.


## 5. Comparing the three fairly

"Fairly" has a precise meaning: **the same data, the same folds, the same
metric, the same protocol** for every candidate. `cross_validated_accuracy` uses
`StratifiedKFold(n_splits=5, shuffle=True, random_state=42)`, so all four rows
of the table below are scored on the identical five partitions of the same 1,760
rows. Comparing a model cross-validated on one seed against another on a
different seed measures the seeds as much as the models.

Two further habits the table encodes:

* **Report the spread, not just the mean.** A gap between two models smaller
  than their fold-to-fold standard deviation is not yet a gap.
* **Keep the baseline in the table.** Every accuracy is meaningless without the
  4.55% floor beside it.


In [10]:
candidates = {
    "baseline (most_frequent)": get_baseline_model(),
    "logistic regression": get_logistic_regression(),
    "knn (k=5)": get_knn(),
    "naive bayes": get_naive_bayes(),
}

rows = []
for name, model in candidates.items():
    outcome = cross_validated_accuracy(make_pipeline(model), X_train, y_train)
    rows.append(
        {
            "model": name,
            **{f"fold {i}": round(score, 4) for i, score in enumerate(outcome["scores"], start=1)},
            "mean": round(outcome["mean"], 4),
            "std": round(outcome["std"], 4),
        }
    )

results = pd.DataFrame(rows).set_index("model")
results

,fold 1,fold 2,fold 3,fold 4,fold 5,mean,std
model,,,,,,,
baseline (most_frequent),0.0455,0.0455,0.0455,0.0455,0.0455,0.0455,0.0000
logistic regression,0.9659,0.9574,0.9688,0.9716,0.9773,0.9682,0.0066
knn (k=5),0.9830,0.9716,0.9460,0.9631,0.9631,0.9653,0.0121
naive bayes,1.0000,0.9972,0.9886,0.9915,0.9972,0.9949,0.0042


The table above is the raw record: five fold scores per model, plus their mean
and standard deviation. The next cell derives the two columns that make it
readable — how far each model sits above the 4.55% floor, and its error rate
(`1 - accuracy`), which is the sharper way to compare models that are all close
to 100%.


In [11]:
baseline_accuracy = results.loc["baseline (most_frequent)", "mean"]
summary = results[["mean", "std"]].copy()
summary["vs baseline"] = (summary["mean"] - baseline_accuracy).round(4)
summary["error rate"] = (1 - summary["mean"]).round(4)

print(summary.to_string())
print("\nbest this week:", summary.drop(index="baseline (most_frequent)")["mean"].idxmax())

                            mean     std  vs baseline  error rate
model                                                            
baseline (most_frequent)  0.0455  0.0000       0.0000      0.9545
logistic regression       0.9682  0.0066       0.9227      0.0318
knn (k=5)                 0.9653  0.0121       0.9198      0.0347
naive bayes               0.9949  0.0042       0.9494      0.0051

best this week: naive bayes


### Reading the table

* **All three beat the baseline by more than 90 percentage points.** That is the
  first thing to check and the only thing that was ever in doubt: the seven
  features carry real information about which crop suits a field, and each
  algorithm extracted it.
* **Naive Bayes wins, at ~99.5%.** The simplest, fastest, most obviously
  misspecified model of the three is the most accurate. Week 2 explains why: the
  crops occupy compact, well-separated blobs in feature space, and "one Gaussian
  per crop per feature" is an almost perfect description of exactly that shape.
  The lesson is not "naive Bayes is best" but "model complexity is not a
  ranking; fit between the model's assumptions and the data's shape is".
* **Logistic regression (~96.8%) and KNN (~96.5%) are not separated by this
  experiment.** Their means differ by ~0.3 points while KNN's fold standard
  deviation alone is ~1.2 points. Declaring a winner there would be reading
  noise; distinguishing them properly needs repeated cross-validation or a
  statistical test, which is Week 6.
* **Look at error rates, not just accuracies.** Naive Bayes' 0.5% error against
  logistic regression's 3.2% is a six-fold difference in mistakes, which the
  accuracy column understates.

### When would you prefer KNN over logistic regression?

* When the class regions are **curved or fragmented** — KNN follows any shape,
  logistic regression only cuts straight.
* When the feature count is **small** and rows are plentiful, so distances stay
  meaningful and the lookup stays cheap.
* When training must be **instant** and new labelled rows arrive constantly:
  adding data to KNN is appending to a list, with no refit.

Prefer logistic regression when you need the model to be **explainable** (Week
7), when prediction must be fast or the model small (176 numbers versus 1,760
stored rows), when there are **many features**, or when calibrated probabilities
matter.


## 6. The result, and what it licenses

> ### Week 5 result — 5-fold stratified CV on the 1,760 training rows
>
> | Model | CV accuracy | vs. 4.55% baseline |
> | --- | --- | --- |
> | Gaussian naive Bayes | **99.49%** (±0.42) | +94.9 points |
> | Logistic regression | 96.82% (±0.66) | +92.3 points |
> | KNN, k = 5 | 96.53% (±1.21) | +92.0 points |
> | `most_frequent` baseline | 4.55% (±0.00) | — |

The current best is **Gaussian naive Bayes at 99.49%**, and every model here
clears the Week 4 floor by a margin far larger than any fold-to-fold wobble, so
"they beat the baseline" is a safe claim in a way that "naive Bayes beats
logistic regression by 2.7 points" is only *probably* safe and "logistic
regression beats KNN by 0.3 points" is not safe at all.

Three cautions to carry into Week 6:

* **These are untuned defaults.** `C = 1.0` and `k = 5` were chosen by
  scikit-learn, not by evidence. Week 6 searches properly, and the ranking may
  change.
* **No test-set number has been quoted.** `data/processed/test.csv` is still
  unopened; these are cross-validated *training* scores, used for choosing
  between models, and the final honest number comes in Week 8.
* **A 99.5% ceiling is a property of this dataset, not a sign of skill.** Week 2
  showed the crops are nearly separable. Weeks 6-8 are worth doing anyway,
  because how you tune, compare and evaluate is what transfers to data that is
  not this obliging.


In [12]:
best_model = summary.drop(index="baseline (most_frequent)")["mean"].idxmax()
best_accuracy = summary.loc[best_model, "mean"]

print(f"BEST SO FAR : {best_model} at {best_accuracy:.4f} ({best_accuracy:.2%})")
print(f"BASELINE    : {baseline_accuracy:.4f} ({baseline_accuracy:.2%})")
print(f"protocol    : {DEFAULT_CV_FOLDS}-fold stratified CV, seed {DEFAULT_RANDOM_STATE},"
      " on data/processed/train.csv")

# Guard rails: these must hold for the conclusions above to be true.
assert baseline_accuracy < 0.06                       # Week 4's floor, unchanged
assert summary.drop(index="baseline (most_frequent)")["mean"].min() > 0.90
assert best_model == "naive bayes"
assert len(train) == 1_760                            # the test set was never touched

BEST SO FAR : naive bayes at 0.9949 (99.49%)
BASELINE    : 0.0455 (4.55%)
protocol    : 5-fold stratified CV, seed 42, on data/processed/train.csv


## 7. What this week produced, and what it deliberately did not

**Produced**

* `get_logistic_regression()`, `get_knn()` and `get_naive_bayes()` in
  `src/models/classical_models.py`, all returning unfitted estimators.
* The `fit(X_train, y_train)` / `predict(X_test)` loop, named once and reused by
  every model from here on.
* A four-row results table — three real models against the Week 4 baseline, on
  identical folds — to be **extended** in Weeks 6, 7 and 8.
* The current leader: Gaussian naive Bayes, 99.49%.

**Not produced, on purpose**

* No tuned hyperparameter. `C = 1.0`, `k = 5` and `var_smoothing = 1e-9` are
  defaults; searching is **Week 6**.
* No ensemble — no random forest, no gradient boosting, no voting classifier —
  **Week 6**.
* No statement about which features drive a prediction — **Week 7**.
* No precision, recall, F1 or confusion matrix — **Week 8**.
* No test-set score. `data/processed/test.csv` remains unopened.
